# Build For_fine_tuning answers with local Qwen3.5-4B

Set SMOKE_TEST = True in the setup cell to process the first two topic groups (four independent Mid/Low requests). Set it to False for all 250 groups. High answers are copied unchanged. Readability is checked against each original target; longer answers are allowed to explain retained information. Thinking remains disabled.


In [8]:
from pathlib import Path
import gc, json, os, re, time
import pandas as pd
import textstat
import torch
from rouge_score import rouge_scorer
from transformers import AutoModelForCausalLM, AutoTokenizer

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

CSV_PATH = ROOT / "newly_generated_answers" / "explicit_answers_local_knowledge.csv"
SOURCE_COLUMN = "Answer_only"
OUTPUT_COLUMN = "For_fine_tuning"
MODEL_ID = "qwen/qwen3.5-4b"

SMOKE_TEST = False  # True: first 2 topic groups (4 Mid/Low targets); False: all 250 groups
SMOKE_GROUPS = 2

MAX_NEW_TOKENS = 2048
TEMPERATURE, TOP_P, TOP_K = 0.3, 0.9, 20
MAX_ATTEMPTS, SEED = 3, 42
FK_TOLERANCE = 1.0
WORD_MIN_RATIO = 0.80
WORD_MAX_RATIO = 1.80
HIGH_WORD_MAX_RATIO = 1.30
COVERAGE_MIN = 0.50
LOAD_IN_4BIT = True

assert CSV_PATH.exists(), CSV_PATH
print("root:", ROOT)
print("csv :", CSV_PATH.relative_to(ROOT))
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print("gpu :", gpu.name, f"{gpu.total_memory / 1024**3:.1f} GiB")


root: /home/anirban/Desktop/Research/qwen_before_after_audit_package_fixed_v2/qwen_before_after_audit
csv : newly_generated_answers/explicit_answers_local_knowledge.csv
cuda: True
gpu : NVIDIA GeForce RTX 5060 Ti 15.6 GiB


## 1. Resolve the verified local Qwen3.5-4B snapshot


In [9]:
def resolve_snapshot() -> Path:
    override = os.getenv("QWEN_LOCAL_MODEL_PATH")
    if override:
        candidates = [Path(override).expanduser().resolve()]
    else:
        roots = []
        if os.getenv("HF_HUB_CACHE"):
            roots.append(Path(os.environ["HF_HUB_CACHE"]).expanduser())
        if os.getenv("HF_HOME"):
            roots.append(Path(os.environ["HF_HOME"]).expanduser() / "hub")
        roots.append(Path.home() / ".cache" / "huggingface" / "hub")
        candidates = []
        for root in roots:
            directory = root / "models--qwen--qwen3.5-4b" / "snapshots"
            if directory.exists():
                candidates.extend(sorted(
                    (path for path in directory.iterdir() if path.is_dir()),
                    key=lambda path: path.stat().st_mtime,
                    reverse=True,
                ))

    required = {
        "config.json", "tokenizer_config.json", "tokenizer.json",
        "model.safetensors.index.json",
        "model.safetensors-00001-of-00002.safetensors",
        "model.safetensors-00002-of-00002.safetensors",
    }
    for candidate in candidates:
        if candidate.is_dir() and all((candidate / name).exists() for name in required):
            return candidate
    raise FileNotFoundError("Set QWEN_LOCAL_MODEL_PATH to a complete local qwen/qwen3.5-4b snapshot.")


LOCAL_MODEL_PATH = resolve_snapshot()
config = json.loads((LOCAL_MODEL_PATH / "config.json").read_text())
assert config.get("model_type") == "qwen3_5"
print("model:", MODEL_ID)
print("local:", LOCAL_MODEL_PATH)


model: qwen/qwen3.5-4b
local: /home/anirban/.cache/huggingface/hub/models--qwen--qwen3.5-4b/snapshots/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a


## 2. Load the CSV and create 250 High/Mid/Low groups


In [10]:
frame = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)
required = {
    "prompt_uid", "ID", "Domain", "Topic", "Topic_Details", "Source_Law",
    "Condition", "Literacy_Level", SOURCE_COLUMN,
}
assert not (required - set(frame.columns)), sorted(required - set(frame.columns))
assert len(frame) == 750
assert frame["prompt_uid"].is_unique
assert frame["Condition"].eq("Explicit").all()
assert frame[SOURCE_COLUMN].str.strip().ne("").all()

if OUTPUT_COLUMN not in frame.columns:
    frame.insert(frame.columns.get_loc(SOURCE_COLUMN) + 1, OUTPUT_COLUMN, "")

LEVELS = ("High Literacy", "Mid Literacy", "Low Literacy")
groups = []
for topic_id, rows in frame.groupby("ID", sort=False):
    assert len(rows) == 3
    levels = {row.Literacy_Level: row for row in rows.itertuples()}
    assert set(levels) == set(LEVELS)
    high, mid, low = (levels[level] for level in LEVELS)
    groups.append({
        "ID": topic_id,
        "Domain": high.Domain,
        "Topic": high.Topic,
        "Topic_Details": high.Topic_Details,
        "Source_Law": high.Source_Law,
        "high": high.Answer_only,
        "mid": mid.Answer_only,
        "low": low.Answer_only,
        "index": {"high": high.Index, "mid": mid.Index, "low": low.Index},
        "mid_fk": float(textstat.flesch_kincaid_grade(mid.Answer_only)),
        "low_fk": float(textstat.flesch_kincaid_grade(low.Answer_only)),
        "mid_words": len(mid.Answer_only.split()),
        "low_words": len(low.Answer_only.split()),
    })
    frame.at[high.Index, OUTPUT_COLUMN] = high.Answer_only

assert len(groups) == 250
print("rows:", len(frame), "| groups:", len(groups))


rows: 750 | groups: 250


## 3. Content-preserving prompt

The High answer supplies content coverage. The original Mid and Low answers supply only their target register and readability. Correctness remains more important than maximizing similarity metrics.


In [11]:
SYSTEM_PROMPT = (
    "You create content-preserving plain-language answers for supervised fine-tuning. "
    "Adapt language without reducing access to important information. "
    "Return only valid JSON. Never output reasoning or thinking tags."
)

PROMPT = """Create one improved {level} literacy answer.

CORE PRINCIPLE
Adapt the language, not the user's access to information.

CONTENT RULES
- Use the High answer as the main content-coverage reference.
- Use the original {level} answer only as the readability and register reference.
- Preserve relevant facts, named entities, laws, organizations, technical terms, numbers,
  deadlines, thresholds, conditions, exceptions, safeguards, risks, rights, and steps.
- Explain difficult terms instead of deleting them.
- Do not introduce facts unsupported by the topic, context, source, or supplied answers.
- Do not preserve obvious speculation or irrelevant detail merely to increase entity count.
- Preserve qualifications and uncertainty and remain consistent with the High answer.
- Factual correctness is the first priority.

{level} TARGET
Flesch-Kincaid grade: {target_fk:.2f}; allowed: {fk_min:.2f} to {fk_max:.2f}.
Original word count: {target_words}; allowed range: {word_min} to {word_max}.
Use extra space when needed to explain retained facts. Shorten sentences, not the fact set.
Use one main idea per sentence. Prefer full stops to long clauses or semicolons.
Aim for {sentence_words} words per sentence. Define long technical names briefly.
{style}
Keep and explain important names, numbers, rights, exceptions, and safety details.
Never sound childish or patronizing.

OPTIMIZE IN THIS ORDER
1. Correctness and source consistency.
2. Important and actionable information retention.
3. Semantic similarity and logical entailment with the High answer.
4. Relevant entity coverage.
5. ROUGE-Lsum overlap when compatible with natural plain language.
6. Target readability and allowed length.

Use plain paragraphs, with no Markdown tables, headings, or emphasis. Retain precise
legal conditions and exceptions. Do not repeat reference headings merely to copy wording.

Do not mention the task, references, literacy evaluation, metrics, or rewriting.
Return only one JSON field:
{{"improved_answer": "..."}}

TOPIC
ID: {ID}
Domain: {Domain}
Topic: {Topic}
Details: {Topic_Details}
Source: {Source_Law}

HIGH CONTENT REFERENCE
<high_answer>
{high}
</high_answer>

{level} REGISTER REFERENCE
<target_answer>
{original}
</target_answer>
{feedback}
"""


def bounds(group: dict, key: str) -> tuple[int, int]:
    original_words = group[f"{key}_words"]
    high_words = len(group["high"].split())
    return (
        max(1, round(original_words * WORD_MIN_RATIO)),
        max(round(original_words * WORD_MAX_RATIO), round(high_words * HIGH_WORD_MAX_RATIO)),
    )


def make_prompt(group: dict, key: str, feedback: str = "") -> str:
    assert key in ("mid", "low")
    target_fk, target_words = group[f"{key}_fk"], group[f"{key}_words"]
    word_min, word_max = bounds(group, key)
    return PROMPT.format(
        ID=group["ID"], Domain=group["Domain"], Topic=group["Topic"],
        Topic_Details=group["Topic_Details"], Source_Law=group["Source_Law"],
        high=group["high"], original=group[key], level=key.title(),
        target_fk=target_fk, fk_min=target_fk - FK_TOLERANCE,
        fk_max=target_fk + FK_TOLERANCE, target_words=target_words,
        word_min=word_min, word_max=word_max,
        style=(
            "Use clear explanations, moderate sentences, and define technical terms."
            if key == "mid" else
            "Use short sentences, common words, direct explanations, and step-by-step organization."
        ),
        feedback=feedback,
        sentence_words="12-16" if key == "mid" else "8-12",
    )


example = next(group for group in groups if group["ID"] == "DIG_13")
for key in ("mid", "low"):
    preview = make_prompt(example, key)
    print(f"\n{key.upper()} prompt ({len(preview):,} characters):\n{preview[:2500]}")



MID prompt (8,727 characters):
Create one improved Mid literacy answer.

CORE PRINCIPLE
Adapt the language, not the user's access to information.

CONTENT RULES
- Use the High answer as the main content-coverage reference.
- Use the original Mid answer only as the readability and register reference.
- Preserve relevant facts, named entities, laws, organizations, technical terms, numbers,
  deadlines, thresholds, conditions, exceptions, safeguards, risks, rights, and steps.
- Explain difficult terms instead of deleting them.
- Do not introduce facts unsupported by the topic, context, source, or supplied answers.
- Do not preserve obvious speculation or irrelevant detail merely to increase entity count.
- Preserve qualifications and uncertainty and remain consistent with the High answer.
- Factual correctness is the first priority.

Mid TARGET
Flesch-Kincaid grade: 8.22; allowed: 7.22 to 9.22.
Original word count: 418; allowed range: 334 to 752.
Use extra space when needed to explain re

## 4. Load the local model with thinking disabled


In [12]:
# Release a model left in this kernel by an earlier failed/full-precision load.
if "model" in globals():
    del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print(f"VRAM before load: {free_bytes / 1024**3:.2f} GiB free / {total_bytes / 1024**3:.2f} GiB")
    if free_bytes < 5 * 1024**3:
        raise RuntimeError(
            "Less than 5 GiB VRAM is free. Shut down the kernel running the previous "
            "answer-generation notebook, restart this kernel, and run again."
        )

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH, local_files_only=True)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

probe = tokenizer.apply_chat_template(
    [{"role": "user", "content": "test"}],
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)
assert probe.endswith("<think>\n\n</think>\n\n")

kwargs = {
    "dtype": torch.bfloat16,
    "device_map": "auto",
    "local_files_only": True,
    "low_cpu_mem_usage": True,
}
if LOAD_IN_4BIT:
    from transformers import BitsAndBytesConfig
    kwargs.pop("dtype")
    kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

model = AutoModelForCausalLM.from_pretrained(LOCAL_MODEL_PATH, **kwargs)
model.eval()

EOS_IDS = {tokenizer.eos_token_id}
configured = getattr(model.generation_config, "eos_token_id", None)
EOS_IDS.update(configured if isinstance(configured, (list, tuple, set)) else [configured])
im_end = tokenizer.convert_tokens_to_ids("<|im_end|>")
if isinstance(im_end, int) and im_end >= 0:
    EOS_IDS.add(im_end)
EOS_IDS.discard(None)

print(f"loaded {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B parameters")
print("thinking: disabled | weights: 4-bit NF4" if LOAD_IN_4BIT else "thinking: disabled | weights: BF16")
if torch.cuda.is_available():
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print(f"VRAM after load: {free_bytes / 1024**3:.2f} GiB free / {total_bytes / 1024**3:.2f} GiB")


VRAM before load: 14.74 GiB free / 15.56 GiB


Loading weights: 100%|██████████| 426/426 [00:06<00:00, 62.21it/s] 

loaded 2.42B parameters
thinking: disabled | weights: 4-bit NF4
VRAM after load: 11.72 GiB free / 15.56 GiB


## 5. Generate, validate JSON, checkpoint, and resume

Only invalid JSON or an invalid requested JSON schema triggers retries. FK, word count, entity coverage, and ROUGE are reported but do not reject an answer. The other acceptance checks are commented out. Thinking remains disabled in the generation template. Each saved nonempty target is skipped on rerun.


In [18]:
ROUGE = rouge_scorer.RougeScorer(["rougeLsum"], use_stemmer=True)
# Conservative proxy: explicit legal references, quantities, and uppercase acronyms.
# Capitalized headings and arbitrary title-case phrases are deliberately excluded.
ENTITY = re.compile(
    r"\b(?:Article|Section|Clause|Recital|Paragraph)\s+\d+[A-Za-z0-9().\-]*"
    r"|[€$£]\s?[\d,]+(?:\.\d+)?"
    r"|\b\d+(?:\.\d+)?\s*%"
    r"|\b\d+\s+(?:days?|weeks?|months?|years?|hours?|minutes?)\b"
    r"|\b(?:19|20)\d{2}\b",
    re.IGNORECASE,
)
ACRONYM = re.compile(r"\b[A-Z]{2,}(?:[-/][A-Z0-9]+)*\b")
NON_ENTITIES = {"HIGH", "MID", "LOW", "NOTE", "IMPORTANT", "SUMMARY", "CONCLUSION", "THE", "AND", "OR"}


def entity_set(text: str) -> set[str]:
    numeric = {re.sub(r"\s+", " ", m.group()).lower().rstrip(".,;:") for m in ENTITY.finditer(text)}
    acronyms = {m.group().lower() for m in ACRONYM.finditer(text) if m.group() not in NON_ENTITIES}
    return numeric | acronyms


def coverage(reference: str, candidate: str) -> float:
    ref = entity_set(reference)
    candidate_normalized = re.sub(r"\s+", " ", candidate).lower()
    matched = sum(
        bool(re.search(r"(?<!\w)" + re.escape(entity) + r"(?!\w)", candidate_normalized))
        for entity in ref
    )
    return 1.0 if not ref else matched / len(ref)


def required_coverage(group: dict, key: str) -> float:
    # Keep at least the original target's coverage and the configured floor.
    return max(COVERAGE_MIN, coverage(group["high"], group[key]))


def parse_json_answer(raw: str, key: str) -> str | None:
    """Accept any JSON object keys whose values contain nonempty answer strings."""
    parse_json_answer.last_error = "No JSON object found"
    if not raw:
        parse_json_answer.last_error = "Empty model output"
        return None
    decoder = json.JSONDecoder(strict=False)
    for position, character in enumerate(raw):
        if character != "{":
            continue
        try:
            obj, _ = decoder.raw_decode(raw[position:])
        except json.JSONDecodeError as error:
            parse_json_answer.last_error = f"JSON syntax: {error.msg} (position {error.pos})"
            continue
        if not isinstance(obj, dict):
            continue
        # JSON preserves field order. Keep every direct, nonempty text value.
        texts = [
            value.strip()
            for value in obj.values()
            if isinstance(value, str) and value.strip()
        ]
        if texts:
            return "\n\n".join(texts)
        parse_json_answer.last_error = (
            f"JSON parsed, but no nonempty string values; keys={list(obj)}"
        )
    return None


def generate(group: dict, key: str, feedback: str) -> tuple[str, bool, int]:
    rendered = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": make_prompt(group, key, feedback)},
        ],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    encoded = tokenizer(rendered, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.inference_mode():
        output = model.generate(
            **encoded,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            top_k=TOP_K,
            pad_token_id=tokenizer.pad_token_id,
        )
    ids = output[0, encoded["input_ids"].shape[1]:].tolist()
    stop = next((i for i, token in enumerate(ids) if token in EOS_IDS), len(ids))
    return tokenizer.decode(ids[:stop], skip_special_tokens=True).strip(), stop >= MAX_NEW_TOKENS, stop


def assess(group: dict, key: str, text: str | None, truncated: bool = False) -> dict:
    if text is None:
        return {"accepted": False, "valid": False, "failures": ["invalid JSON"]}
    # Truncation alone does not trigger a retry when the requested JSON parsed successfully.
    # if truncated:
    #     return {"accepted": False, "valid": False, "failures": ["truncated"]}
    word_min, word_max = bounds(group, key)
    fk = float(textstat.flesch_kincaid_grade(text))
    words = len(text.split())
    cov = coverage(group["high"], text)
    minimum_coverage = required_coverage(group, key)
    failures = []
    # if abs(fk - group[f"{key}_fk"]) > FK_TOLERANCE:
    #     failures.append(f"FK {fk:.2f} outside {group[f'{key}_fk'] - FK_TOLERANCE:.2f}-{group[f'{key}_fk'] + FK_TOLERANCE:.2f}")
    # if not word_min <= words <= word_max:
    #     failures.append(f"words {words} outside {word_min}-{word_max}")
    # if cov + 1e-9 < minimum_coverage:
    #     failures.append(f"entity coverage {cov:.0%} below {minimum_coverage:.0%}")
    return {
        "valid": True, "fk": fk, "words": words, "coverage": cov,
        "rouge": ROUGE.score(group["high"], text)["rougeLsum"].fmeasure,
        "accepted": not failures, "failures": failures,
    }


def feedback_for(group: dict, key: str, text: str | None, scores: dict, truncated: bool) -> str:
    return (
        '\n\nRETRY: The previous response did not contain valid JSON in the required form. '
        'Return exactly one JSON object with one nonempty string field: "improved_answer". '
        'Escape quotation marks inside the answer. Return JSON only.'
    )


def save_atomic():
    temporary = CSV_PATH.with_suffix(CSV_PATH.suffix + ".tmp")
    try:
        frame.to_csv(temporary, index=False, lineterminator="\n")
        os.replace(temporary, CSV_PATH)
    finally:
        if temporary.exists():
            temporary.unlink()


def complete_targets() -> set[tuple[str, str]]:
    done = set()
    for group in selected_groups:
        for key in ("mid", "low"):
            text = str(frame.at[group["index"][key], OUTPUT_COLUMN])
            if text.strip() and assess(group, key, text)["accepted"]:
                done.add((group["ID"], key))
    return done


selected_groups = groups[:SMOKE_GROUPS] if SMOKE_TEST else groups
TARGET_TOTAL = len(selected_groups) * 2

# Reload saved targets so rerunning this cell resumes independently for each level.
latest = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)
assert latest["prompt_uid"].tolist() == frame["prompt_uid"].tolist(), "CSV row order changed"
pd.testing.assert_series_equal(latest[SOURCE_COLUMN], frame[SOURCE_COLUMN])
if OUTPUT_COLUMN in latest.columns:
    frame[OUTPUT_COLUMN] = latest[OUTPUT_COLUMN]
for group in groups:
    frame.at[group["index"]["high"], OUTPUT_COLUMN] = group["high"]

done = complete_targets()
todo = [(group, key) for group in selected_groups for key in ("mid", "low") if (group["ID"], key) not in done]
print(f"SMOKE_TEST={SMOKE_TEST} | selected groups={len(selected_groups)}")
print(f"resume | complete targets {len(done)}/{TARGET_TOTAL} | remaining {len(todo)}")

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

failures = []
failed_json_outputs = {}  # Inspect failed_json_outputs[(topic_id, level, attempt)] after a run.
started = time.time()

try:
    for number, (group, key) in enumerate(todo, 1):
        accepted = None
        feedback = ""
        for attempt in range(1, MAX_ATTEMPTS + 1):
            try:
                raw, truncated, tokens = generate(group, key, feedback)
            except torch.cuda.OutOfMemoryError as error:
                gc.collect()
                torch.cuda.empty_cache()
                raise RuntimeError(
                    f"CUDA OOM on {group['ID']} {key}. Restart this kernel and "
                    "make sure no other notebook kernel is holding GPU memory."
                ) from error

            candidate = parse_json_answer(raw, key)
            scores = assess(group, key, candidate, truncated)
            if scores["valid"]:
                print(
                    f"{number:3d}/{len(todo)} | {group['ID']:9s} | {key.upper():3s} | "
                    f"attempt {attempt} | {tokens:4d} tok | coverage {scores['coverage']:.2f} | "
                    f"FK {scores['fk']:.1f} | words {scores['words']} | "
                    f"{'accepted' if scores['accepted'] else 'retry: ' + '; '.join(scores['failures'])}", flush=True,
                )
            else:
                reason = parse_json_answer.last_error
                failed_json_outputs[(group["ID"], key, attempt)] = {
                    "raw": raw, "reason": reason, "truncated": truncated, "tokens": tokens,
                }
                print(
                    f"{number:3d}/{len(todo)} | {group['ID']} | {key.upper()} | "
                    f"attempt {attempt} | {tokens} tok | invalid JSON: {reason} | "
                    f"truncated={truncated}", flush=True,
                )
                print("  raw preview:", repr(raw[:240]), flush=True)
            if scores["accepted"]:
                accepted = candidate
                break
            feedback = feedback_for(group, key, candidate, scores, truncated)

        if accepted:
            frame.at[group["index"][key], OUTPUT_COLUMN] = accepted
            done.add((group["ID"], key))
            # Persist each response immediately; a failed Low cannot lose a passing Mid.
            save_atomic()
            print(f"  .. saved target {len(done)}/{TARGET_TOTAL}", flush=True)
        else:
            failures.append(f"{group['ID']}|{key.title()}")
finally:
    # Preserve passing targets on interruption or any unexpected exception.
    save_atomic()

print(f"pass complete | {len(done)}/{TARGET_TOTAL} targets | {(time.time() - started) / 60:.1f} minutes")
if failures:
    print("Rerun this cell to retry these targets:", failures)
    raise RuntimeError(f"{len(failures)} targets did not pass all gates")


SMOKE_TEST=False | selected groups=250
resume | complete targets 495/500 | remaining 5
  1/5 | DIG_09    | LOW | attempt 1 |  184 tok | coverage 0.50 | FK 10.5 | words 159 | accepted
  .. saved target 496/500
  2/5 | DIG_14    | MID | attempt 1 |  428 tok | coverage 0.20 | FK 15.2 | words 326 | accepted
  .. saved target 497/500
  3/5 | DIG_14    | LOW | attempt 1 |   39 tok | coverage 0.20 | FK 4.9 | words 26 | accepted
  .. saved target 498/500
  4/5 | SEC_06    | LOW | attempt 1 |  293 tok | coverage 0.00 | FK 10.3 | words 249 | accepted
  .. saved target 499/500
  5/5 | SAF_06    | MID | attempt 1 |  561 tok | coverage 0.71 | FK 15.2 | words 448 | accepted
  .. saved target 500/500
pass complete | 500/500 targets | 0.7 minutes


## 6. Final validation


In [19]:
result = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)
assert len(result) == 750
assert result["prompt_uid"].tolist() == frame["prompt_uid"].tolist()
assert OUTPUT_COLUMN in result.columns
assert result.columns.get_loc(OUTPUT_COLUMN) == result.columns.get_loc(SOURCE_COLUMN) + 1
selected_indices = [g["index"][k] for g in selected_groups for k in ("high", "mid", "low")]
assert result.loc[selected_indices, OUTPUT_COLUMN].str.strip().ne("").all(), "Selected targets are incomplete"
# assert not result[OUTPUT_COLUMN].str.contains("<think>|</think>", regex=True).any()

high = result["Literacy_Level"].eq("High Literacy")
assert result.loc[high, OUTPUT_COLUMN].equals(result.loc[high, SOURCE_COLUMN])

records = []
for group in selected_groups:
    for key in ("mid", "low"):
        text = result.at[group["index"][key], OUTPUT_COLUMN]
        scores = assess(group, key, text)
        records.append({
            "ID": group["ID"], "level": key.title(), "passes": scores["accepted"],
            "fk": scores["fk"], "words": scores["words"],
            "coverage": scores["coverage"], "rouge_lsum": scores["rouge"],
        })

quality = pd.DataFrame(records)
assert len(quality) == TARGET_TOTAL
assert quality["passes"].all()
print(f"Validation passed | SMOKE_TEST={SMOKE_TEST} | {len(selected_groups)} groups | {TARGET_TOTAL} targets")
print(quality.groupby("level")[["fk", "words", "coverage", "rouge_lsum"]].mean().round(3))
quality.head(10)


Validation passed | SMOKE_TEST=False | 250 groups | 500 targets
           fk    words  coverage  rouge_lsum
level                                       
Low     8.977  228.576     0.562       0.364
Mid    15.435  294.700     0.719       0.642


,ID,level,passes,fk,words,coverage,rouge_lsum
0,DIG_01,Mid,True,18.043374,326,0.666667,0.549020
1,DIG_01,Low,True,6.689255,230,0.500000,0.228866
2,DIG_02,Mid,True,16.280641,195,1.000000,0.674115
3,DIG_02,Low,True,8.102626,107,1.000000,0.237668
4,DIG_03,Mid,True,15.799343,274,0.222222,0.438149
5,DIG_03,Low,True,7.449482,276,0.111111,0.252852
6,DIG_04,Mid,True,15.732107,175,1.000000,0.488889
7,DIG_04,Low,True,8.126596,188,1.000000,0.395639
8,DIG_05,Mid,True,17.780818,428,0.800000,0.777365
9,DIG_05,Low,True,14.028255,408,0.600000,0.700855
